# SMA Crossover (10-50) on SPY
## Strategy Brief
This strategy uses a simple moving average (SMA) crossover to generate trading signals on the SPY ETF. When the 10-day SMA crosses above the 50-day SMA, it signals a potential uptrend, prompting a buy signal. Conversely, when the 10-day SMA crosses below the 50-day SMA, it indicates a potential downtrend, suggesting a sell signal. The strategy aims to capture medium-term trends in the SPY, potentially outperforming a buy-and-hold approach.
## References
- https://www.sma-america.com/

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the parameters for our SMA crossover strategy, including the short and long SMA periods and the ticker symbol for SPY.

In [ ]:
TICKER = 'SPY'
START_DATE = '2010-01-01'
SHORT_WINDOW = 10
LONG_WINDOW = 50

### PHASE 2 - Data Exploration
We will download historical price data for SPY using yfinance, calculate the 10-day and 50-day SMAs, and plot these indicators over the price data.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download historical data for SPY
data = yf.download(TICKER, start=START_DATE)

# Calculate SMAs
data['SMA10'] = data['Close'].rolling(window=SHORT_WINDOW).mean()
data['SMA50'] = data['Close'].rolling(window=LONG_WINDOW).mean()

# Plot closing price and SMAs
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='Close Price')
plt.plot(data['SMA10'], label='10-Day SMA')
plt.plot(data['SMA50'], label='50-Day SMA')
plt.title('SPY Price and SMA Crossover')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
Here, we define the trading signals based on SMA crossovers. A buy signal is generated when the 10-day SMA crosses above the 50-day SMA, and a sell signal is generated when the opposite occurs.

In [ ]:
# Generate signals
data['Signal'] = 0
# Buy signal
data.loc[data['SMA10'] > data['SMA50'], 'Signal'] = 1
# Sell signal
data.loc[data['SMA10'] < data['SMA50'], 'Signal'] = -1

# Positions based on signals
data['Position'] = data['Signal'].shift(1)

### PHASE 4 - Coding & Backtesting
We will calculate daily returns, apply the strategy's positions to these returns, and plot the resulting equity curve.

In [ ]:
# Calculate daily returns
data['Market_Returns'] = data['Close'].pct_change()
# Strategy returns
data['Strategy_Returns'] = data['Market_Returns'] * data['Position']

# Calculate equity curve
data['Equity_Curve'] = (1 + data['Strategy_Returns']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity_Curve'], label='Strategy Equity Curve')
plt.title('Equity Curve for SMA Crossover Strategy')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
We will evaluate the strategy's performance using metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. These will be compared against a buy-and-hold strategy.

In [ ]:
from scipy.stats import norm

# Calculate performance metrics
def calculate_performance(data):
    # CAGR
    total_return = data['Equity_Curve'].iloc[-1] - 1
    n_years = (data.index[-1] - data.index[0]).days / 365.25
    cagr = (1 + total_return) ** (1 / n_years) - 1
    
    # Sharpe Ratio
    sharpe_ratio = data['Strategy_Returns'].mean() / data['Strategy_Returns'].std() * np.sqrt(252)
    
    # Sortino Ratio
    downside_returns = data.loc[data['Strategy_Returns'] < 0, 'Strategy_Returns']
    sortino_ratio = data['Strategy_Returns'].mean() / downside_returns.std() * np.sqrt(252)
    
    # Max Drawdown
    roll_max = data['Equity_Curve'].cummax()
    daily_drawdown = data['Equity_Curve'] / roll_max - 1.0
    max_drawdown = daily_drawdown.cummin().min()
    
    # Calmar Ratio
    calmar_ratio = cagr / abs(max_drawdown)
    
    return pd.Series({'CAGR': cagr, 'Sharpe Ratio': sharpe_ratio, 'Sortino Ratio': sortino_ratio, 'Calmar Ratio': calmar_ratio, 'Max Drawdown': max_drawdown})

# Calculate metrics
strategy_metrics = calculate_performance(data)

# Buy and Hold Strategy
buy_hold_return = (data['Close'].iloc[-1] / data['Close'].iloc[0]) - 1
buy_hold_cagr = (1 + buy_hold_return) ** (1 / n_years) - 1
buy_hold_metrics = pd.Series({'CAGR': buy_hold_cagr, 'Sharpe Ratio': np.nan, 'Sortino Ratio': np.nan, 'Calmar Ratio': np.nan, 'Max Drawdown': np.nan})

# Compare strategies
comparison = pd.DataFrame({'SMA Crossover': strategy_metrics, 'Buy and Hold': buy_hold_metrics})
comparison

### PHASE 6 - Deploy & Monitor
We will create a function to download the last 60 days of SPY data, compute today's SMA crossover signal, and print the recommended position.

In [ ]:
def get_latest_signal():
    # Download last 60 days of data
    recent_data = yf.download(TICKER, period='60d')
    
    # Calculate SMAs
    recent_data['SMA10'] = recent_data['Close'].rolling(window=SHORT_WINDOW).mean()
    recent_data['SMA50'] = recent_data['Close'].rolling(window=LONG_WINDOW).mean()
    
    # Determine the latest signal
    if recent_data['SMA10'].iloc[-1] > recent_data['SMA50'].iloc[-1]:
        print('Buy Signal')
    elif recent_data['SMA10'].iloc[-1] < recent_data['SMA50'].iloc[-1]:
        print('Sell Signal')
    else:
        print('No Signal')

get_latest_signal()